# COMP9517 — HOG + SVM Pipeline
**Method:** HOG (Histogram of Oriented Gradients) + LinearSVC  
**Advanced direction:** Robustness to test-time image degradation  

Run cells top-to-bottom. Results save to Google Drive automatically.

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/comp9517'
DATA_ROOT  = f'{DRIVE_ROOT}/dataset'
CODE_DIR   = f'{DRIVE_ROOT}/code'
RESULTS    = f'{DRIVE_ROOT}/results'

import os
os.makedirs(RESULTS, exist_ok=True)
print('Drive mounted. DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# ── Cell 2: Pull latest code from GitHub ──────────────────────────────────
# Replace with your actual GitHub repo URL after you create it
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/comp9517-hog.git'

import os
if os.path.exists('/content/comp9517-hog'):
    !cd /content/comp9517-hog && git pull
else:
    !git clone {GITHUB_REPO} /content/comp9517-hog

# Add to Python path so imports work
import sys
sys.path.insert(0, '/content/comp9517-hog')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────
!pip install -q scikit-image scikit-learn joblib matplotlib pandas pillow opencv-python-headless

In [ ]:
# ── Cell 4: Verify dataset structure ──────────────────────────────────────
import os
from pathlib import Path

for split in ['train', 'val', 'test']:
    split_dir = Path(DATA_ROOT) / split
    if split_dir.exists():
        n_species = len([d for d in split_dir.iterdir() if d.is_dir()])
        n_images  = sum(1 for _ in split_dir.rglob('*.jpg'))
        print(f'{split:6s}: {n_species:4d} species, {n_images:7d} images')
    else:
        print(f'{split:6s}: NOT FOUND at {split_dir}')

In [ ]:
# ── Cell 5: Train HOG + LinearSVC ─────────────────────────────────────────
# Expected runtime: ~5-15 min on Colab CPU for 500 species
!python /content/comp9517-hog/train_hog_classifier.py \
    --train   {DATA_ROOT}/train \
    --val     {DATA_ROOT}/val \
    --test    {DATA_ROOT}/test \
    --out_dir {RESULTS}/hog \
    --svm_c   0.1

In [ ]:
# ── Cell 6: Display results summary ──────────────────────────────────────
import json
with open(f'{RESULTS}/hog/results.json') as f:
    results = json.load(f)

print('\n=== HOG + LinearSVC Results ===')
for split_key in ['svm_val', 'svm_test', 'rf_test']:
    if split_key not in results:
        continue
    r = results[split_key]
    label = {'svm_val': 'LinearSVC (val)', 'svm_test': 'LinearSVC (test)', 'rf_test': 'Random Forest (test)'}[split_key]
    print(f'\n{label}')
    print(f'  Top-1 accuracy : {r["top1_acc"]:.4f}')
    print(f'  Top-5 accuracy : {r["top5_acc"]:.4f}')
    print(f'  Macro precision: {r["macro_precision"]:.4f}')
    print(f'  Macro recall   : {r["macro_recall"]:.4f}')
    print(f'  Macro F1       : {r["macro_f1"]:.4f}')
    if 'train_time_s' in r:
        print(f'  Train time     : {r["train_time_s"]:.1f}s')
    print(f'  Infer time     : {r["infer_time_s"]:.1f}s')

print('\nTop-10 most confused species pairs (LinearSVC):')
for count, true_cls, pred_cls in results['svm_test']['most_confused']:
    print(f'  {count:3d}x  {true_cls}  →  {pred_cls}')

In [ ]:
# ── Cell 7: Show confusion matrix ─────────────────────────────────────────
from IPython.display import Image
Image(f'{RESULTS}/hog/svm_confusion_matrix.png')

## Robustness Study
Applies 5 degradation types × 5 severity levels to the **test set only**.  
No retraining. Expected runtime: ~30-60 min.

In [ ]:
# ── Cell 8: Run robustness evaluation ─────────────────────────────────────
!python /content/comp9517-hog/evaluate_robustness.py \
    --model   {RESULTS}/hog/hog_svm.joblib \
    --test    {DATA_ROOT}/test \
    --classes {RESULTS}/hog/class_names.json \
    --out_dir {RESULTS}/hog/robustness

In [ ]:
# ── Cell 9: Plot robustness curves ────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'{RESULTS}/hog/robustness/robustness_results.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
clean_top1 = df[df['degradation'] == 'clean']['top1_acc'].values[0]
clean_f1   = df[df['degradation'] == 'clean']['macro_f1'].values[0]

degradations = [d for d in df['degradation'].unique() if d != 'clean']
colours = plt.cm.tab10.colors

for ax, metric, baseline, ylabel in [
    (axes[0], 'top1_acc', clean_top1, 'Top-1 Accuracy'),
    (axes[1], 'macro_f1', clean_f1,   'Macro F1'),
]:
    for i, deg in enumerate(degradations):
        sub = df[df['degradation'] == deg].sort_values('severity')
        ax.plot(sub['severity'], sub[metric], marker='o', label=deg, color=colours[i])
    ax.axhline(baseline, color='black', linestyle='--', linewidth=1.5, label='clean')
    ax.set_xlabel('Severity Level')
    ax.set_ylabel(ylabel)
    ax.set_title(f'HOG+LinearSVC — {ylabel} vs Degradation')
    ax.legend(fontsize=8)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS}/hog/robustness/robustness_curves.png', dpi=150)
plt.show()
print('Saved robustness_curves.png')

In [ ]:
# ── Cell 10: Robustness summary table ─────────────────────────────────────
pivot = df[df['degradation'] != 'clean'].pivot_table(
    index='degradation', columns='severity', values='top1_acc'
).round(4)
pivot.columns = [f'sev={c}' for c in pivot.columns]
pivot['clean_baseline'] = clean_top1
pivot['drop_at_sev5'] = clean_top1 - pivot['sev=5']
print('Top-1 accuracy by degradation and severity:')
print(pivot.to_string())
pivot.to_csv(f'{RESULTS}/hog/robustness/summary_table.csv')

## Optional: HOG Ablation Study
Sweeps `pixels_per_cell` and `orientations`. Run after baseline to find best HOG config.

In [ ]:
# ── Cell 11: HOG ablation (optional, ~30-45 min) ──────────────────────────
!python /content/comp9517-hog/ablation_hog.py \
    --train   {DATA_ROOT}/train \
    --val     {DATA_ROOT}/val \
    --out_dir {RESULTS}/ablation